# Exploration with query helpers

In [1]:
import sys
from dataclasses import asdict
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
from logstore.db import connect
from logstore.repositories import LogRepository, LogShapeRepository
from datetime import datetime

conn = connect()
logs = LogRepository(conn)
log_shapes = LogShapeRepository(conn)

## Log shape stats

In [2]:
stats = log_shapes.stats()
df_stats = pd.DataFrame([asdict(s) for s in stats])
print(f"{len(df_stats)} log shapes, {df_stats.n_logs.sum():,} total logs")
df_stats.head(20)

141 log shapes, 411,483 total logs


,component,state,template,n_logs,n_nodes,first_seen,last_seen,examples
0,switch_module,error,Linkerror event interval expired,80895,16,2004-01-14 16:43:12,2005-05-18 16:40:59,"[Linkerror event interval expired, Linkerror e..."
1,node,temperature,ambient=<N>,54119,257,2003-12-26 14:58:30,2006-04-30 07:01:00,"[ambient=24, ambient=36, ambient=31]"
2,gige,temperature,warning,49157,8,2003-12-26 12:36:12,2006-04-30 11:23:24,"[warning, warning, warning]"
3,gige,temperature,normal,45079,7,2003-12-26 12:41:12,2006-04-30 11:28:24,"[normal, normal, normal]"
4,switch_module,fan,Fan speeds ( <N> <N> <N> <N> <N> <N> ),21559,1,2004-09-19 04:42:40,2006-04-10 06:03:16,"[Fan speeds ( 3552 3552 3391 4245 3515 3497 ),..."
5,node,status,running,18834,257,2003-12-26 18:42:21,2006-04-27 14:58:04,"[running, running, running]"
6,node,status,configured out,16027,256,2003-08-05 11:01:31,2006-04-26 22:18:00,"[configured out, configured out, configured out]"
7,switch_module,error,link errors remain current,13538,16,2005-05-19 15:57:24,2006-04-30 09:10:07,"[link errors remain current, link errors remai..."
8,action,start,boot (command <N>),13075,256,2003-12-26 18:24:10,2006-04-24 21:18:09,"[boot (command 1873), boot (command 3005), b..."
9,action,start,wait (command <N>),12689,256,2003-12-26 18:27:10,2006-04-24 21:21:54,"[wait (command 1873), wait (command 3008), w..."


### Log shapes by log count

In [3]:
df_stats.sort_values("n_logs", ascending=False)[["component", "state", "template", "n_logs", "n_nodes"]].head(30)

,component,state,template,n_logs,n_nodes
0,switch_module,error,Linkerror event interval expired,80895,16
1,node,temperature,ambient=<N>,54119,257
2,gige,temperature,warning,49157,8
3,gige,temperature,normal,45079,7
4,switch_module,fan,Fan speeds ( <N> <N> <N> <N> <N> <N> ),21559,1
5,node,status,running,18834,257
6,node,status,configured out,16027,256
7,switch_module,error,link errors remain current,13538,16
8,action,start,boot (command <N>),13075,256
9,action,start,wait (command <N>),12689,256


Let's investigate a random instance from each of the top 10 log shapes.

In [4]:
temp = df_stats.iloc[0]
logs.last_by_log_shape().head(20)

TypeError: LogRepository.last_by_log_shape() missing 2 required positional arguments: 'key' and 'n'

## logs_around: a node's neighbourhood

In [5]:
ts = datetime(2004, 2, 27, 12, 0, 0)
around = logs.around("node-0", ts, before_s=3600, after_s=3600)
print(f"{len(around)} logs for node-0 around {ts}")
pd.DataFrame([asdict(r) for r in around]).head(20)

2 logs for node-0 around 2004-02-27 12:00:00


,id,event_id,node,component,state,time,flag,message,bad_time,ts,original_log,log_shape_id,n_copies
0,391958,168035,node-0,unix.hw,net.niff.up,1.077886e+09,1,NIFF: node node-0 has detected an available ne...,False,2004-02-27 12:46:01,168035 node-0 unix.hw net.niff.up 1077885961 1...,138,1
1,391959,168036,node-0,unix.hw,net.niff.up,1.077886e+09,1,NIFF: node node-0 has detected an available ne...,False,2004-02-27 12:46:01,168036 node-0 unix.hw net.niff.up 1077885961 1...,137,1


## node_timeline: collapsed view

In [6]:
start = datetime(2004, 2, 27, 0, 0, 0)
end = datetime(2004, 2, 28, 0, 0, 0)
tl = logs.node_timeline("node-0", start, end)
print(f"{len(tl)} entries for node-0 on the busiest day")
pd.DataFrame([asdict(e) for e in tl])

12 entries for node-0 on the busiest day


,component,state,template,log_shape_id,first_ts,last_ts,count
0,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available ne...,139,2004-02-27 09:35:04,2004-02-27 09:35:04,1
1,action,start,boot (command <N>),7,2004-02-27 09:57:03,2004-02-27 09:57:03,1
2,action,abort,boot (command <N>),41,2004-02-27 10:09:27,2004-02-27 10:09:27,1
3,action,start,boot (command <N>),7,2004-02-27 10:31:28,2004-02-27 10:31:28,1
4,action,abort,boot (command <N>),41,2004-02-27 10:55:17,2004-02-27 10:55:17,1
5,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available ne...,138,2004-02-27 12:46:01,2004-02-27 12:46:01,1
6,unix.hw,net.niff.up,NIFF: node <NODE> has detected an available ne...,137,2004-02-27 12:46:01,2004-02-27 12:46:01,1
7,action,start,boot (command <N>),7,2004-02-27 14:50:43,2004-02-27 14:50:43,1
8,action,start,wait (command <N>),6,2004-02-27 14:55:11,2004-02-27 14:55:11,1
9,action,start,boot (command <N>),7,2004-02-27 17:20:01,2004-02-27 17:20:01,1


## A different node for comparison

In [7]:
tl2 = logs.node_timeline("Interconnect-0N00", start, end)
print(f"{len(tl2)} entries for Interconnect-0N00 on 2004-02-27")
pd.DataFrame([asdict(e) for e in tl2])

1 entries for Interconnect-0N00 on 2004-02-27


,component,state,template,log_shape_id,first_ts,last_ts,count
0,switch_module,error,Linkerror event interval expired,110,2004-02-27 09:54:58,2004-02-27 17:33:26,613


## A filesystem incident

In [8]:
# find a fdmn.panic event as an anchor
panics = conn.execute(
    "SELECT l.id, l.ts, l.node, l.message FROM logs l "
    "JOIN log_shapes s ON l.log_shape_id = s.id "
    "WHERE s.state = 'fdmn.panic' ORDER BY l.ts LIMIT 5"
).fetchall()
for p in panics:
    print(p)

(41299, '2003-12-30 05:37:31', 'node-D7', 'ServerFileSystem: An ServerFileSystem domain panic has occurred on storage1223')
(41298, '2003-12-30 05:37:32', 'node-D7', 'ServerFileSystem: An ServerFileSystem domain panic has occurred on storage1211')
(41297, '2003-12-30 05:37:48', 'node-D7', 'ServerFileSystem: An ServerFileSystem domain panic has occurred on storage1241')
(41300, '2004-02-26 14:01:03', 'node-D0', 'ServerFileSystem: An ServerFileSystem domain panic has occurred on storage344')
(41301, '2004-02-26 14:01:11', 'node-D0', 'ServerFileSystem: An ServerFileSystem domain panic has occurred on storage323')


In [9]:
if panics:
    _, panic_ts, panic_node, _ = panics[0]
    ts = datetime.fromisoformat(panic_ts)
    around = logs.around(panic_node, ts, before_s=7200, after_s=7200)
    print(f"{len(around)} logs for {panic_node} around the first fdmn.panic")
    pd.DataFrame([asdict(r) for r in around])

167 logs for node-D7 around the first fdmn.panic


In [10]:
if panics:
    tl3 = logs.node_timeline(panic_node, datetime.fromisoformat(panic_ts),
                             datetime(ts.year, ts.month, ts.day, 23, 59, 59))
    print(f"Timeline for {panic_node} from the panic to end of day")
    pd.DataFrame([asdict(e) for e in tl3])

Timeline for node-D7 from the panic to end of day


In [11]:
conn.close()